# Local Upload: `la.json` to Hugging Face

로컬 `la.json` 파일을 Hugging Face dataset repo `HYUNJINI/jssp_validation_all_v1`의 `validation_data/la.json` 경로로 업로드합니다.


In [ ]:
!pip -q install -U huggingface_hub hf_transfer


## 설정


In [ ]:
import os
from pathlib import Path
from huggingface_hub import HfApi, login

CANDIDATE_ROOTS = [
    r'C:\\Users\\User\\Desktop\\LLM_JSSP_masking',
    '/mnt/c/Users/User/Desktop/LLM_JSSP_masking',
]
PROJECT_ROOT = next((p for p in CANDIDATE_ROOTS if Path(p).exists()), CANDIDATE_ROOTS[-1])
PROJECT_ROOT = str(Path(PROJECT_ROOT).expanduser().resolve())
root = Path(PROJECT_ROOT)
print('PROJECT_ROOT:', root)

######################
# HUGGING FACE TOKEN #
######################
HF_TOKEN = ''
######################

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = ''

if not HF_TOKEN:
    raise ValueError('HF_TOKEN is empty. Fill the block above or set HF_TOKEN env var.')

login(token=HF_TOKEN, add_to_git_credential=False)
print('HF login ready')
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

REPO_ID = 'HYUNJINI/jssp_validation_all_v1'
REPO_TYPE = 'dataset'
LOCAL_FILE = root / 'la.json'
PATH_IN_REPO = 'validation_data/la.json'

print('LOCAL_FILE:', LOCAL_FILE)
print('PATH_IN_REPO:', PATH_IN_REPO)
assert LOCAL_FILE.exists(), f'la.json not found: {LOCAL_FILE}'
api = HfApi(token=HF_TOKEN)

preview = LOCAL_FILE.read_text(encoding='utf-8')[:300]
print('preview:', preview)
print('file size (MB):', round(LOCAL_FILE.stat().st_size / (1024**2), 4))


## 업로드 실행


In [ ]:
api.create_repo(repo_id=REPO_ID, repo_type=REPO_TYPE, private=False, exist_ok=True)
result = api.upload_file(
    path_or_fileobj=str(LOCAL_FILE),
    path_in_repo=PATH_IN_REPO,
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
)
print('upload complete:', result)


## 업로드 확인


In [ ]:
files = api.list_repo_files(repo_id=REPO_ID, repo_type=REPO_TYPE)
matches = [f for f in files if f == PATH_IN_REPO or f.endswith('/la.json')]
print('match count:', len(matches))
for f in matches:
    print(' -', f)
